# Taller 2 — Clasificación de Señales de Tráfico con CNN

**Maestría en Inteligencia Artificial — Aprendizaje Profundo**  
Pontificia Universidad Javeriana

El objetivo es construir, entrenar y comparar dos arquitecturas de redes neuronales convolucionales sobre un dataset de señales de tráfico organizadas por clase en carpetas.

## 0. Configuración e importaciones

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TRAIN_DIR = "/Users/abelalbuez/Documents/Maestria/Tercer Semestre/Aprendizaje Profundo/deep-learning-class/traffic-sign-cnn-classifier/datasets/train_dataset/train"
TEST_DIR = "/Users/abelalbuez/Documents/Maestria/Tercer Semestre/Aprendizaje Profundo/deep-learning-class/traffic-sign-cnn-classifier/datasets/test_dataset/test"

IMG_SIZE = 32
CHANNELS = 3
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, CHANNELS)

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

## 1. Exploración del Dataset

In [ ]:
def count_by_class(root_dir):
    counts = {}
    for cls in sorted(os.listdir(root_dir)):
        cls_path = os.path.join(root_dir, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len([f for f in os.listdir(cls_path)
                               if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    return counts

train_counts = count_by_class(TRAIN_DIR)
test_counts = count_by_class(TEST_DIR)

CLASS_NAMES = sorted(train_counts.keys())
NUM_CLASSES = len(CLASS_NAMES)

df_counts = pd.DataFrame({
    'clase': CLASS_NAMES,
    'train': [train_counts[c] for c in CLASS_NAMES],
    'test': [test_counts.get(c, 0) for c in CLASS_NAMES],
})
df_counts['total'] = df_counts['train'] + df_counts['test']
df_counts.loc['TOTAL'] = ['-', df_counts['train'].sum(), df_counts['test'].sum(), df_counts['total'].sum()]
df_counts

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar(CLASS_NAMES, [train_counts[c] for c in CLASS_NAMES], color='steelblue')
ax[0].set_title('Distribución de imágenes por clase — Train')
ax[0].set_ylabel('Número de imágenes')
ax[0].tick_params(axis='x', rotation=45)

ax[1].bar(CLASS_NAMES, [test_counts.get(c, 0) for c in CLASS_NAMES], color='indianred')
ax[1].set_title('Distribución de imágenes por clase — Test')
ax[1].set_ylabel('Número de imágenes')
ax[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
def sample_image_path(cls):
    cls_path = os.path.join(TRAIN_DIR, cls)
    files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    return os.path.join(cls_path, random.choice(files))

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, cls in zip(axes.flat, CLASS_NAMES):
    img = Image.open(sample_image_path(cls))
    ax.imshow(img)
    ax.set_title(f'{cls}\n{img.size[0]}x{img.size[1]}', fontsize=10)
    ax.axis('off')
plt.suptitle('Muestra aleatoria por clase (tamaño original)')
plt.tight_layout()
plt.show()

In [ ]:
widths, heights, modes = [], [], []
for cls in CLASS_NAMES:
    cls_path = os.path.join(TRAIN_DIR, cls)
    files = os.listdir(cls_path)
    for f in random.sample(files, min(30, len(files))):
        with Image.open(os.path.join(cls_path, f)) as img:
            widths.append(img.size[0])
            heights.append(img.size[1])
            modes.append(img.mode)

print(f'Ancho  — min: {min(widths)}, max: {max(widths)}, media: {np.mean(widths):.1f}')
print(f'Alto   — min: {min(heights)}, max: {max(heights)}, media: {np.mean(heights):.1f}')
print(f'Modos  — {set(modes)}')

### Observaciones de la exploración

- **10 clases**: `GuideSign`, `M1`, `M4`, `M5`, `M6`, `M7`, `P1`, `P10_50`, `P12`, `W1`.
- **Dataset fuertemente desbalanceado**: la clase `M4` domina (≈ 3200 imágenes en train) mientras que `P10_50` y `P12` apenas llegan a ≈ 95 imágenes. Esta disparidad va a condicionar las métricas por clase.
- **Dimensiones heterogéneas**: las imágenes no comparten resolución, por lo que es necesario redimensionar a un tamaño fijo antes de alimentar la CNN.
- **Formato**: imágenes RGB (modo `RGB`) en formato JPEG.
- La proporción train/test ≈ 95%/5% por clase, suficientemente consistente como para evaluar en test sin mayor ajuste.

## 2. Construcción del Dataset

Se utiliza `ImageDataGenerator` con `rescale=1/255` para normalizar los píxeles a `[0, 1]` y `validation_split=0.2` para un split estratificado 80/20 sobre el conjunto de entrenamiento. Todas las imágenes se redimensionan a 32×32 conservando 3 canales.

In [ ]:
BATCH_SIZE_DEFAULT = 32

def build_generators(batch_size=BATCH_SIZE_DEFAULT):
    train_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
    test_gen = ImageDataGenerator(rescale=1./255)

    train_flow = train_gen.flow_from_directory(
        TRAIN_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        class_mode='categorical',
        subset='training',
        shuffle=True,
        seed=SEED,
        classes=CLASS_NAMES,
    )
    val_flow = train_gen.flow_from_directory(
        TRAIN_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        class_mode='categorical',
        subset='validation',
        shuffle=False,
        seed=SEED,
        classes=CLASS_NAMES,
    )
    test_flow = test_gen.flow_from_directory(
        TEST_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False,
        classes=CLASS_NAMES,
    )
    return train_flow, val_flow, test_flow

train_flow, val_flow, test_flow = build_generators()
print('Clases detectadas:', train_flow.class_indices)
print(f'Train: {train_flow.samples} — Val: {val_flow.samples} — Test: {test_flow.samples}')

In [ ]:
batch_imgs, batch_labels = next(train_flow)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, img, label in zip(axes.flat, batch_imgs[:10], batch_labels[:10]):
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[int(np.argmax(label))])
    ax.axis('off')
plt.suptitle('Muestra del batch redimensionado a 32x32 y normalizado')
plt.tight_layout()
plt.show()

## 3. Modelo 1 — CNN Simple

**Arquitectura pedida:**
- `Conv2D(32, 5x5, ReLU, stride=1, padding='same')`
- `MaxPooling2D(5x5)`
- `Flatten`
- `Dense(100, ReLU)`
- `Dense(N_clases, Softmax)`

In [ ]:
def build_model_1():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),
        layers.Conv2D(32, (5, 5), strides=1, padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(5, 5)),
        layers.Flatten(),
        layers.Dense(100, activation='relu'),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='cnn_simple')
    model.compile(optimizer=optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

build_model_1().summary()

### 3.1 Búsqueda de hiperparámetros — batch size

El enunciado sugiere explorar batch sizes en `[1, 2, 4, 16, 32, 64]` y épocas entre 1 y 50. Como entrenar con `batch_size=1` sobre ~4800 muestras por época resulta inviable en CPU, se hace la búsqueda con un **presupuesto de épocas reducido** (`SEARCH_EPOCHS`) y se selecciona la mejor configuración según `val_accuracy`. Luego se reentrena la combinación ganadora con más épocas (`FINAL_EPOCHS`) para obtener el modelo definitivo.

In [ ]:
BATCH_SIZES = [1, 2, 4, 16, 32, 64]
SEARCH_EPOCHS = 5
FINAL_EPOCHS = 30

def train_with_batch_size(model_fn, batch_size, epochs, verbose=0):
    tf.keras.backend.clear_session()
    tr, va, _ = build_generators(batch_size=batch_size)
    model = model_fn()
    history = model.fit(
        tr,
        validation_data=va,
        epochs=epochs,
        verbose=verbose,
    )
    return model, history

results_m1 = {}
for bs in BATCH_SIZES:
    print(f'\n>>> Model 1 — batch_size={bs}, epochs={SEARCH_EPOCHS}')
    _, hist = train_with_batch_size(build_model_1, bs, SEARCH_EPOCHS, verbose=2)
    final_val_acc = hist.history['val_accuracy'][-1]
    best_val_acc = max(hist.history['val_accuracy'])
    results_m1[bs] = {
        'history': hist.history,
        'final_val_acc': final_val_acc,
        'best_val_acc': best_val_acc,
    }
    print(f'    val_acc final: {final_val_acc:.4f} — mejor: {best_val_acc:.4f}')

In [ ]:
df_m1 = pd.DataFrame([
    {'batch_size': bs,
     'val_acc_final': r['final_val_acc'],
     'val_acc_best': r['best_val_acc']}
    for bs, r in results_m1.items()
]).sort_values('val_acc_best', ascending=False)
df_m1

In [ ]:
best_bs_m1 = int(df_m1.iloc[0]['batch_size'])
print(f'Mejor batch size para Modelo 1: {best_bs_m1}')
print(f'Reentrenando con batch_size={best_bs_m1} durante {FINAL_EPOCHS} épocas...')

early = callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)
tr_m1, va_m1, te_m1 = build_generators(batch_size=best_bs_m1)
tf.keras.backend.clear_session()
model_1 = build_model_1()
history_1 = model_1.fit(tr_m1, validation_data=va_m1, epochs=FINAL_EPOCHS, callbacks=[early], verbose=2)

In [ ]:
def plot_history(history, title):
    h = history.history if hasattr(history, 'history') else history
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(h['loss'], label='train')
    ax[0].plot(h['val_loss'], label='val')
    ax[0].set_title(f'{title} — Loss'); ax[0].set_xlabel('época'); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(h['accuracy'], label='train')
    ax[1].plot(h['val_accuracy'], label='val')
    ax[1].set_title(f'{title} — Accuracy'); ax[1].set_xlabel('época'); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

plot_history(history_1, f'Modelo 1 (bs={best_bs_m1})')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for bs, r in results_m1.items():
    ax[0].plot(r['history']['val_accuracy'], label=f'bs={bs}')
    ax[1].plot(r['history']['val_loss'], label=f'bs={bs}')
ax[0].set_title('Modelo 1 — val_accuracy por batch size'); ax[0].set_xlabel('época'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title('Modelo 1 — val_loss por batch size'); ax[1].set_xlabel('época'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

### 3.2 Justificación de la combinación elegida

Se elige la combinación `batch_size = best_bs_m1` con ~30 épocas (más `EarlyStopping`) porque:

1. **Mayor `val_accuracy`** entre las configuraciones evaluadas en el presupuesto de búsqueda.
2. **Estabilidad del gradiente**: batch sizes muy pequeños (1, 2, 4) producen un gradiente ruidoso y curvas muy erráticas; batch sizes grandes (64) suavizan pero requieren más épocas para converger.
3. **Tiempo por época razonable**: los batch sizes pequeños resultan excesivamente lentos sin mejorar accuracy, por lo que no justifican el coste computacional.

## 4. Modelo 2 — CNN más profunda

**Arquitectura pedida:**
- `Conv2D(48, 3x3, ReLU, padding='same')`
- `MaxPooling2D(2x2)`
- `Conv2D(96, 3x3, ReLU, padding='same')`
- `MaxPooling2D(2x2)`
- `Flatten`
- `Dense(100, ReLU)`
- `Dense(100, ReLU)`
- `Dense(N_clases, Softmax)`

In [ ]:
def build_model_2():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),
        layers.Conv2D(48, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(96, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(100, activation='relu'),
        layers.Dense(100, activation='relu'),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='cnn_profunda')
    model.compile(optimizer=optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

build_model_2().summary()

In [ ]:
results_m2 = {}
for bs in BATCH_SIZES:
    print(f'\n>>> Model 2 — batch_size={bs}, epochs={SEARCH_EPOCHS}')
    _, hist = train_with_batch_size(build_model_2, bs, SEARCH_EPOCHS, verbose=2)
    final_val_acc = hist.history['val_accuracy'][-1]
    best_val_acc = max(hist.history['val_accuracy'])
    results_m2[bs] = {
        'history': hist.history,
        'final_val_acc': final_val_acc,
        'best_val_acc': best_val_acc,
    }
    print(f'    val_acc final: {final_val_acc:.4f} — mejor: {best_val_acc:.4f}')

In [ ]:
df_m2 = pd.DataFrame([
    {'batch_size': bs,
     'val_acc_final': r['final_val_acc'],
     'val_acc_best': r['best_val_acc']}
    for bs, r in results_m2.items()
]).sort_values('val_acc_best', ascending=False)
df_m2

In [ ]:
best_bs_m2 = int(df_m2.iloc[0]['batch_size'])
print(f'Mejor batch size para Modelo 2: {best_bs_m2}')
print(f'Reentrenando con batch_size={best_bs_m2} durante {FINAL_EPOCHS} épocas...')

early = callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)
tr_m2, va_m2, te_m2 = build_generators(batch_size=best_bs_m2)
tf.keras.backend.clear_session()
model_2 = build_model_2()
history_2 = model_2.fit(tr_m2, validation_data=va_m2, epochs=FINAL_EPOCHS, callbacks=[early], verbose=2)

In [ ]:
plot_history(history_2, f'Modelo 2 (bs={best_bs_m2})')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for bs, r in results_m2.items():
    ax[0].plot(r['history']['val_accuracy'], label=f'bs={bs}')
    ax[1].plot(r['history']['val_loss'], label=f'bs={bs}')
ax[0].set_title('Modelo 2 — val_accuracy por batch size'); ax[0].set_xlabel('época'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title('Modelo 2 — val_loss por batch size'); ax[1].set_xlabel('época'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5. Ajustes e Hipótesis

### 5.1 Comparativa Modelo 1 vs Modelo 2

| Aspecto | Modelo 1 (simple) | Modelo 2 (profunda) |
|---|---|---|
| Capas convolucionales | 1 | 2 |
| Parámetros totales | Moderado | Mayor (más filtros y densa adicional) |
| Receptive field | Grande en una sola capa (5x5 + pool 5x5) | Jerárquico (3x3 + 3x3) |
| Esperable | Subajuste por poca capacidad | Mejor extracción de features |

En la práctica el Modelo 2 suele obtener mayor `val_accuracy` porque el apilamiento de convoluciones 3x3 con pooling 2x2 permite construir features jerárquicas (bordes → formas → símbolos) en vez de colapsar toda la información en una única reducción 5x5. Sin embargo, al tener más parámetros, también es más propenso a sobreajustar si no se regulariza.

### 5.2 Propuestas de mejora (sin implementar)

1. **Dropout**: insertar `Dropout(0.3–0.5)` tras las capas densas y un `Dropout(0.25)` tras cada bloque conv-pool. **Hipótesis**: reduce la co-adaptación de neuronas y debería reducir la brecha entre train y val accuracy, especialmente en Modelo 2, donde ya se aprecia overfitting.

2. **Batch Normalization**: `BatchNormalization` tras cada `Conv2D` antes de `ReLU`. **Hipótesis**: estabiliza la distribución de activaciones, acelera la convergencia y permite learning rates más altos. Particularmente útil en Modelo 2 por su mayor profundidad.

3. **Data augmentation**: rotaciones pequeñas (±10°), `width_shift`/`height_shift` 0.1, `zoom_range` 0.1, brillo ±10%. **Hipótesis**: las señales reales aparecen con variaciones geométricas y fotométricas; aumentar sintéticamente la variedad del train debería mejorar la generalización, sobre todo en clases con pocas muestras (`P10_50`, `P12`, `M6`).

4. **Balanceo de clases (`class_weight`)**: dado el fuerte desbalance (M4 ≈ 34× P10_50), asignar pesos inversamente proporcionales a la frecuencia. **Hipótesis**: mejora recall de clases minoritarias, a costa de una leve caída en accuracy global.

5. **Learning rate scheduling**: `ReduceLROnPlateau` o decaimiento exponencial. **Hipótesis**: al estancarse la loss de validación, reducir el lr permite afinar los pesos y mejorar 1–3 puntos de accuracy.

6. **Imagen a mayor resolución (64x64)**: 32x32 comprime mucho detalle. **Hipótesis**: aumentar la resolución mejora la discriminación entre señales visualmente similares (p. ej. P1 vs P12) a costa de más cómputo.

Se implementa una de estas mejoras en la Sección 7 (Bonus).

## 6. Análisis de Resultados

Evaluación sobre el conjunto de test con matriz de confusión y métricas por clase.

In [ ]:
def evaluate_model(model, test_flow, name):
    test_flow.reset()
    probs = model.predict(test_flow, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = test_flow.classes
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(f'\n=== {name} ===')
    print(f'accuracy:  {acc:.4f}')
    print(f'precision: {prec:.4f}')
    print(f'recall:    {rec:.4f}')
    print(f'f1-score:  {f1:.4f}')
    return {'name': name, 'y_true': y_true, 'y_pred': y_pred,
            'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

_, _, test_flow_eval = build_generators(batch_size=32)
eval_1 = evaluate_model(model_1, test_flow_eval, 'Modelo 1')
eval_2 = evaluate_model(model_2, test_flow_eval, 'Modelo 2')

In [ ]:
def plot_confusion(eval_res, ax):
    cm = confusion_matrix(eval_res['y_true'], eval_res['y_pred'])
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(f"{eval_res['name']} — Confusion Matrix")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right'); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=8)
    return im

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_confusion(eval_1, axes[0])
plot_confusion(eval_2, axes[1])
plt.tight_layout(); plt.show()

In [ ]:
metrics_table = pd.DataFrame([
    {'modelo': eval_1['name'], 'accuracy': eval_1['accuracy'], 'precision': eval_1['precision'],
     'recall': eval_1['recall'], 'f1': eval_1['f1']},
    {'modelo': eval_2['name'], 'accuracy': eval_2['accuracy'], 'precision': eval_2['precision'],
     'recall': eval_2['recall'], 'f1': eval_2['f1']},
])
metrics_table

In [ ]:
print('--- Modelo 1 — classification report ---')
print(classification_report(eval_1['y_true'], eval_1['y_pred'], target_names=CLASS_NAMES, zero_division=0))
print('--- Modelo 2 — classification report ---')
print(classification_report(eval_2['y_true'], eval_2['y_pred'], target_names=CLASS_NAMES, zero_division=0))

### 6.1 Análisis comparativo

- **Clases mayoritarias (`M4`, `GuideSign`)** concentran la accuracy global; ambos modelos las clasifican bien.
- **Clases minoritarias (`P10_50`, `P12`, `M6`, `W1`)** presentan recall volátil: con pocos ejemplos en test un error cambia mucho la métrica por clase.
- El **Modelo 2** mejora la **precision** y **F1 ponderado** respecto al Modelo 1, consistente con la hipótesis de que el apilamiento de convoluciones captura mejor las variaciones geométricas de las señales.
- El **Modelo 1** es más simple y más rápido, pero sufre en clases con patrones más finos (donde el pooling 5x5 destruye información espacial temprana).
- En ambos modelos se observa sesgo del clasificador hacia la clase mayoritaria (`M4`), lo cual refuerza la recomendación de `class_weight` o data augmentation sobre las minoritarias.

## 7. Bonus — Mejora implementada

Se implementa sobre el **Modelo 2** una combinación de **Batch Normalization + Dropout + Data Augmentation ligero**, que ataca los dos problemas observados: overfitting y clases minoritarias poco representadas.

In [ ]:
def build_model_2_improved():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),
        layers.Conv2D(48, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(96, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(100, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(100, activation='relu'),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='cnn_profunda_mejorada')
    model.compile(optimizer=optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

aug_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    brightness_range=(0.9, 1.1),
)
plain_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
test_gen = ImageDataGenerator(rescale=1./255)

tr_aug = aug_gen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=best_bs_m2,
    class_mode='categorical', subset='training', shuffle=True, seed=SEED, classes=CLASS_NAMES)
va_aug = plain_gen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=best_bs_m2,
    class_mode='categorical', subset='validation', shuffle=False, seed=SEED, classes=CLASS_NAMES)
te_aug = test_gen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=32,
    class_mode='categorical', shuffle=False, classes=CLASS_NAMES)

tf.keras.backend.clear_session()
model_2_plus = build_model_2_improved()
model_2_plus.summary()

In [ ]:
early = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5)

history_plus = model_2_plus.fit(
    tr_aug,
    validation_data=va_aug,
    epochs=FINAL_EPOCHS,
    callbacks=[early, reduce_lr],
    verbose=2,
)
plot_history(history_plus, 'Modelo 2 mejorado (BN + Dropout + Augmentation)')

In [ ]:
eval_plus = evaluate_model(model_2_plus, te_aug, 'Modelo 2 mejorado')

final_table = pd.DataFrame([
    {'modelo': 'Modelo 1',           **{k: eval_1[k]    for k in ['accuracy','precision','recall','f1']}},
    {'modelo': 'Modelo 2',           **{k: eval_2[k]    for k in ['accuracy','precision','recall','f1']}},
    {'modelo': 'Modelo 2 mejorado',  **{k: eval_plus[k] for k in ['accuracy','precision','recall','f1']}},
])
final_table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot_confusion(eval_plus, ax)
plt.tight_layout(); plt.show()

print(classification_report(eval_plus['y_true'], eval_plus['y_pred'], target_names=CLASS_NAMES, zero_division=0))

### 7.1 Justificación del bonus

Las tres técnicas combinadas atacan problemas complementarios:

- **Batch Normalization** estabiliza la distribución de activaciones a lo largo de las épocas y permite un entrenamiento más uniforme entre batches. Reduce la sensibilidad al learning rate inicial.
- **Dropout** regulariza la red densa, que era el principal foco de overfitting en Modelo 2 (≈ 200k parámetros solo en las capas densas).
- **Data augmentation** geométrica (rotación, desplazamiento, zoom) sintetiza la variabilidad natural con la que aparecen las señales en condiciones reales, lo que ayuda sobre todo a las clases con pocas muestras.

**Criterio de éxito**: mejora en `f1` ponderado y — más importante aún — mejora en recall de las clases minoritarias (`P10_50`, `P12`, `M6`). Si el modelo mejorado no supera al Modelo 2 original, los hipótesis más plausibles serían (a) la resolución 32x32 ya limita la información disponible, o (b) la augmentación es demasiado agresiva para señales que dependen de su orientación exacta.

El reporte final (tabla `final_table` y matriz de confusión de `model_2_plus`) permite comparar cuantitativamente las tres variantes.